In [58]:
import numpy as np
import pandas as pd
import scipy.stats as st
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.stats.proportion import proportions_ztest, proportion_effectsize
from statsmodels.stats.power import NormalIndPower

np.random.seed(42)

# Task 1: Chi-Square Goodness-of-Fit Test

**Scenario:** A mobile-game studio expects that players choose among four character classes in equal proportions (25 % each). After a recent update, they collected the following counts from 400 new players:

| Class | Warrior | Mage | Rogue | Healer |
|-------|---------|------|-------|--------|
| Count | 120     | 85   | 110   | 85     |

1. Store the observed counts in a NumPy array.
2. Compute the expected counts under the null hypothesis of equal preference.
3. Run `st.chisquare(observed, f_exp=expected)`.
4. Report the χ² statistic, degrees of freedom, and p-value.
5. At α = 0.05, state whether you reject or fail to reject H₀.
6. In a Markdown cell, answer: "Which class (if any) appears over- or under-represented, and by how much?"



In [59]:
observed_counts  = np.array([120,85,110,85])
observed_counts


array([120,  85, 110,  85])

In [60]:
expected = np.array([100,100,100,100])
expected

array([100, 100, 100, 100])

In [61]:
chi2_stat, p_value  = st.chisquare(observed_counts, f_exp = expected)

In [62]:
chi2_stat

np.float64(9.5)

In [63]:
chi2_stat, p_value 

(np.float64(9.5), np.float64(0.023331360430831553))

In [64]:
print(f"Decrees of freedom is {len(expected)-1}")

Decrees of freedom is 3


At α = 0.05 and p value - 0.023331360430831553, p value < α, therefore  we conclude that there is sufficient evidence to reject H0(Null Hypothesis). 

In [65]:
import pandas as pd

df = pd.DataFrame(
    [observed_counts - expected],
    columns=['Warrior', 'Mage', 'Rogue', 'Healer']
)

df


,Warrior,Mage,Rogue,Healer
0,20,-15,10,-15


Warrior appears **over-represented** by +20 players (120 vs. 100 expected).  
Mage appears **under-represented** by −15 players (85 vs. 100 expected).  
Rogue appears **over-represented** by +10 players (110 vs. 100 expected).  
Healer appears **under-represented** by −15 players (85 vs. 100 expected).


# Task 2: Chi-Square Test of Independence

**Scenario:** A marketing team wants to know whether subscription tier (Free, Basic, Premium) is associated with churn status (Churned, Retained). They collected data from 600 customers:

| | Free | Basic | Premium |
|---------|------|-------|---------|
| Churned | 90   | 60    | 30      |
| Retained| 110  | 140   | 170     |

1. Create the contingency table as a 2 × 3 NumPy array or pandas DataFrame.
2. Run `st.chi2_contingency(table)` and unpack χ², p-value, degrees of freedom, and expected frequencies.
3. Display the expected-frequency table.
4. Report the χ² statistic, df, and p-value.
5. State your decision and a plain-language interpretation (e.g. "There is / is not evidence of an association between subscription tier and churn.").

In [66]:
import pandas as pd

data = {
    "Free": [90, 110],
    "Basic": [60, 140],
    "Premium": [30, 170]
}

df = pd.DataFrame(data, index=["Churned", "Retained"])
df


,Free,Basic,Premium
Churned,90,60,30
Retained,110,140,170


In [67]:
chi2, p_value, dof, expected = st.chi2_contingency(df)

In [68]:
chi2

np.float64(42.85714285714286)

In [69]:
p_value

np.float64(4.939576018831198e-10)

In [70]:
dof

2

In [71]:
expected

array([[ 60.,  60.,  60.],
       [140., 140., 140.]])

In [72]:
print(f"\nChi-square statistic: {chi2:.4f}")
print(f"p-value: {p_value:.6f}")
print(f"Degrees of freedom: {dof}")


Chi-square statistic: 42.8571
p-value: 0.000000
Degrees of freedom: 2


In [73]:
print(f"\nExpected frequencies (if independent):")
expected_df = pd.DataFrame(
    expected, index=df.index, columns=df.columns)
expected_df


Expected frequencies (if independent):


,Free,Basic,Premium
Churned,60.0,60.0,60.0
Retained,140.0,140.0,140.0


There is sufficient evidence to conclude that **subscription tier and churn status are associated (not independent)**.

# Task 3: Compute Cramér's V

Using the χ² statistic and the contingency table from Task 2:

1. Write a function:

```python
def cramers_v(chi2, n, min_dim):
    """
    Returns Cramér's V.

    Parameters
    ----------
    chi2 : float
        Chi-square statistic.
    n : int
        Total number of observations.
    min_dim : int
        min(rows, cols) - 1 of the contingency table.
    """
```

2. Compute Cramér's V for the Task 2 result.
3. Classify the effect size using Cohen's benchmarks for df* = min(rows, cols) − 1:
   - Small ≈ 0.10, Medium ≈ 0.30, Large ≈ 0.50 (for 2 × 3 tables, df* = 1)
4. In a Markdown cell, answer: "Is the relationship between tier and churn weak, moderate, or strong? What does this mean for the marketing team?"

In [74]:
def cramers_v(df, chi2, n):
    chi2 = float(chi2)
    n = int(n)
    min_dim = int(min(df.shape)-1)
    return f"Cramer's V : { np.sqrt( chi2 / (n*min_dim))}"

In [75]:
df_cramers_v = cramers_v(df,42.8571, df.values.sum())
print(df_cramers_v)

Cramer's V : 0.26726110828177


The relationship between subscription tier and churn is **moderate (Cramér’s V ≈ 0.27)**. This means there is a meaningful association: churn behavior differs across tiers. For the marketing team, this suggests that targeting strategies by subscription tier (e.g., focusing on high-churn Free users) could be effective.